# PostgreSQL transactional outbox: a hands-on implementation

**Track:** Production · **Guide:** [`061-transactional-outbox.md`](../guides/concepts/061-transactional-outbox.md)

A transactional outbox writes application data and an event record in one database transaction, then publishes the event asynchronously.

> **Intuition:** The order and its dispatch note go into the same locked ledger; a separate courier can retry delivery.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement PostgreSQL transactional outbox]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Index a saved document into Qdrant after commit, tolerating worker restarts and duplicate event delivery.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Create source and outbox tables

The outbox stores the delivery request in the same database as application data.


In [ ]:
import json
import sqlite3

database = sqlite3.connect(":memory:")
database.execute("PRAGMA foreign_keys = ON")
database.executescript("""
CREATE TABLE documents(id TEXT PRIMARY KEY, body TEXT NOT NULL);
CREATE TABLE outbox(
    id INTEGER PRIMARY KEY,
    event_type TEXT NOT NULL,
    aggregate_id TEXT NOT NULL,
    payload TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'pending',
    attempts INTEGER NOT NULL DEFAULT 0
);
""")

## 2. Write the document and event in one transaction

If either insert fails, the context manager rolls back both writes. Parameterized values prevent SQL injection.


In [ ]:
document_id = "doc-1"
payload = {"document_id": document_id, "body": "Returns within 30 days"}
with database:
    database.execute(
        "INSERT INTO documents(id, body) VALUES (?, ?)",
        (document_id, payload["body"]),
    )
    database.execute(
        "INSERT INTO outbox(event_type, aggregate_id, payload) VALUES (?, ?, ?)",
        ("document.index", document_id, json.dumps(payload)),
    )
print("pending events:", database.execute("SELECT count(*) FROM outbox").fetchone()[0])

## 3. Deliver idempotently and mark processed

A worker may crash after delivery but before marking completion. A stable event/aggregate ID lets the destination upsert safely on retry.


In [ ]:
indexed = {}
def process_event(event_id):
    row = database.execute(
        "SELECT aggregate_id, payload FROM outbox WHERE id=? AND status='pending'",
        (event_id,),
    ).fetchone()
    if row is None:
        return False
    aggregate_id, encoded = row
    event = json.loads(encoded)
    indexed[aggregate_id] = event["body"]  # idempotent upsert
    with database:
        database.execute(
            "UPDATE outbox SET status='processed', attempts=attempts+1 WHERE id=?",
            (event_id,),
        )
    return True

event_id = database.execute("SELECT id FROM outbox").fetchone()[0]
print("delivered:", process_event(event_id))
print("repeat is harmless:", process_event(event_id), "index:", indexed)

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/061-transactional-outbox.md) and the
[learning roadmap](../guides/ROADMAP.md).
